# Imports

**NOTE: Make sure to use the get_properties_environment file to set your conda environment.**

In [ ]:
import os,re,sys
import warnings
warnings.filterwarnings("ignore")
import numpy as np
np.set_printoptions(threshold=sys.maxsize) #print out full arrays
import pandas as pd
from pandas import ExcelWriter
import shutil
# import xlsxwriter

import math
randomstate = 42

from rdkit import Chem                                                                                                                                                                  
from rdkit import RDLogger                                                                                                                                                               
RDLogger.DisableLog('rdApp.*')    
from rdkit.Chem import SDMolSupplier, SDWriter
from rdkit.Chem import Draw
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import SVG, display
from IPython.display import Image, display

import goodvibes.GoodVibes as gv
import goodvibes.thermo as thermo
import goodvibes.io as io
import goodvibes.pes as pes
from morfeus import ConeAngle
from morfeus import Sterimol
import get_properties_functions_m as gp

D3 import failed


##### This code is an amended version of the general code found here (https://github.com/SigmanGroup/GetProperties), for the maintained and general code please refer to to the linked github

## Check Multiplicity

In [2]:
directory = '/Users/theresewild/Sigman Group Dropbox/Therese Wild/NN_Library/dft_library_all/hydride_library_ligands/updated_basis_set/biim_matches_old_basis_set'
errors = os.path.join(directory, 'multiplicity_errors')
os.makedirs(errors, exist_ok=True)
multiplicity_errors = set()
for file in os.listdir(directory):
    if file.endswith('.log'):
        multi = "Charge =  0 Multiplicity = 1"
        count = 0
        path = os.path.join(directory, file)
        with open(path) as f:
            for line in f:
                if multi in line:
                    count += 1

        if count != 5:
            print (file)
            prefix = file.split('_')[0]
            multiplicity_errors.add(prefix)

for file in os.listdir(directory):
    if file.endswith('.log'):
        prefix = file.split('_')[0]
        if prefix.endswith('.log'):
            prefix = file.split('.')[0]
        if prefix in multiplicity_errors:
            src = os.path.join(directory, file)
            dst = os.path.join(errors, file)
            shutil.move(src, dst)

# Atom Inputs Dataframe

### Generate dataframe with atom numbers

In [ ]:
substructure1 = Chem.MolFromSmarts ('[H][Ni]1([H])[N]C=C[N]1') # just atoms normally collected
substructure2 = Chem.MolFromSmarts ('[H][Ni]1([H])[N]CCC[N]1')
substructure3 = Chem.MolFromSmarts ('[H][Ni]1([H])[N]CC[N]1') #N1 is pyridine
metal_atom = "Ni"

In [ ]:
with open("log_ids.txt", "r") as log_file:
    log_ids_a = log_file.readlines()

all_ligands = Chem.SDMolSupplier("molecules.sdf", removeHs = False)
molecules_a = [mol for mol in all_ligands]

sorted_data = sorted(zip(log_ids_a, molecules_a), key=lambda x: x[0].strip())
sorted_log_ids, sorted_molecules = zip(*[(log_id.strip(), mol) for log_id, mol in sorted_data])

log_ids = []
molecules = []
for log_id, molecule in zip(sorted_log_ids, sorted_molecules):
    log_ids.append(log_id)
    molecules.append(molecule)

labeled_mol = dict(zip(log_ids,molecules))
labeled_mol.items()

atom_labels_5 = {}
atom_labels_6 = {}
atom_labels_P = {}
for molecule in labeled_mol.items():
    if molecule[1] is not None:
        submatch = molecule[1].GetSubstructMatches(substructure1)
        if len(submatch) > 0:
            matchlist = list([item for sublist in submatch for item in sublist]) #list of zero-indexed atom numbers 
            match_idx = [x+1 for x in matchlist] #this line changes from 0-indexed to 1-indexed (for Gaussian)
            atom_labels_5[molecule[0]] = match_idx
        elif len(submatch) == 0:
            submatch = molecule[1].GetSubstructMatches(substructure2) 
            if len(submatch) > 0:
                matchlist = list([item for sublist in submatch for item in sublist]) #list of zero-indexed atom numbers 
                match_idx = [x+1 for x in matchlist] #this line changes from 0-indexed to 1-indexed (for Gaussian)
                atom_labels_6[molecule[0]] = match_idx
            if len(submatch) == 0:
                submatch = molecule[1].GetSubstructMatches(substructure3) 
                matchlist = list([item for sublist in submatch for item in sublist]) #list of zero-indexed atom numbers 
                match_idx = [x+1 for x in matchlist] #this line changes from 0-indexed to 1-indexed (for Gaussian)
                atom_labels_P[molecule[0]] = match_idx                
    else:
        print (molecule)
        atom_labels_P[molecule[0]] = 'no data'
        
        
stripped_AL5 = {key[:-4]: value for key, value in atom_labels_5.items()}
stripped_AL6 = {key[:-4]: value for key, value in atom_labels_6.items()}

df5 = pd.DataFrame.from_dict(stripped_AL5, orient='index')
df6 = pd.DataFrame.from_dict(stripped_AL6, orient='index')
errors_df = pd.DataFrame.from_dict(atom_labels_P, orient='index')

In [ ]:
df6.drop(columns=[5], inplace=True)
df5 = pd.concat([df5, df6, errors_df])

In [ ]:
atom_labels = {'log_name': 'log_name',
                0: '-H1',
                1: 'Ni', 
                2: '-H2',
                3: 'N2',
                4: 'C2',
                5: 'C1',
                6: 'N1',}

### Generate labeled dataframe

In [ ]:
all_ligands = df5
all_ligands = all_ligands.reset_index().rename(columns={'index': 'log_name'})
atom_map_df = all_ligands.rename(columns=atom_labels)
atom_map_df = atom_map_df.rename(columns={'index': 'log_name'})
atom_map_df = atom_map_df.sort_values(by='log_name', ascending=True)

### Check For Mismatched Numbering in Ligands

In [ ]:
atom_map_df['log_name'] = atom_map_df['log_name'].map(lambda x: x.lstrip('.').rstrip('.log'))
df = atom_map_df
df['lig_name'] = df['log_name'].str.split('_').str[0]

In [ ]:
#make sure atom numbering is consistent within a ligand ID - must be for atom labeling 
lig_names = df.drop_duplicates(subset=['lig_name'])
lig_names = (list (lig_names['lig_name']))
auto_fix_prefixes = lig_names  

ligands = df.groupby('lig_name')
for prefix, ligand in ligands:
    unique_rows = ligand.drop(columns=['log_name', 'lig_name']).drop_duplicates()

    if len(unique_rows) > 1:
        if (
            prefix in auto_fix_prefixes and
            len(unique_rows) == 2 and
            unique_rows.iloc[0]['Ni'] == unique_rows.iloc[1]['Ni'] and
            (
                (unique_rows.iloc[0]['N1'] == unique_rows.iloc[1]['N2']) and
                (unique_rows.iloc[0]['N2'] == unique_rows.iloc[1]['N1'])
            )
        ):
            print(f"\nAuto-fixing Prefix: {prefix} (Detected swapped N1/N2 with same Ni)")
            chosen_values = unique_rows.iloc[0]
            for col in unique_rows.columns:
                df.loc[df['lig_name'] == prefix, col] = chosen_values[col]
            continue

        print(f"\nFlagged Prefix: {prefix}")
        print("Detected differing sets of values:")

        for i, unique_row in enumerate(unique_rows.iterrows(), start=1):
            print(f"Option {i}:")
            print(unique_row[1].to_dict())

        print(f"{len(unique_rows) + 1}: Leave as is")

        choice = int(input(f"\nEnter the option number (1-{len(unique_rows) + 1}) you want to apply for prefix '{prefix}': ").strip())

        if choice <= len(unique_rows):
            chosen_values = unique_rows.iloc[choice - 1]
            for col in unique_rows.columns:
                df.loc[df['lig_name'] == prefix, col] = chosen_values[col]
        else:
            print(f"Leaving the rows for prefix '{prefix}' as they are.")

df = df.drop(columns=['lig_name'])

### Check that Each Atom is the Right Element

In [ ]:
# just double check that atom is what it says it is in the SDF file + df
sdf_dir = '/Users/theresewild/Sigman Group Dropbox/Therese Wild/NN_Library/dft_library_all/hydride_library_ligands/updated_basis_set/biim_matches_old_basis_set'
for i,row in df.iterrows():
    log_name_base = row['log_name']
    N1_index = row['N1'] - 1 #subtract 1 because mol is 0 indexed
    N2_index = row['N2'] - 1
    C1_index = row['C1'] - 1
    C2_index = row['C2'] - 1
    H1_index = row['-H1'] - 1
    H2_index = row['-H2'] - 1
    
    sdf_name = log_name_base + '.sdf'
    sdf_path = os.path.join(sdf_dir,sdf_name)
    try:
        suppl = Chem.SDMolSupplier(sdf_path, removeHs=False)
        mol = next((m for m in suppl if m is not None), None)
        C2_atomic_num = mol.GetAtomWithIdx(C2_index).GetAtomicNum()
        C1_atomic_num = mol.GetAtomWithIdx(C1_index).GetAtomicNum()
        N2_atomic_num = mol.GetAtomWithIdx(N2_index).GetAtomicNum()
        N1_atomic_num = mol.GetAtomWithIdx(N1_index).GetAtomicNum()
        H1_atomic_num = mol.GetAtomWithIdx(H1_index).GetAtomicNum()
        H2_atomic_num = mol.GetAtomWithIdx(H2_index).GetAtomicNum()

        if C2_atomic_num != 6 or C1_atomic_num != 6 or N2_atomic_num !=7 or N1_atomic_num !=7 or H1_atomic_num !=1 or H2_atomic_num !=1:
            print (f'error in atom numbering for {log_name_base}')
    except:
        print (f'bad sdf file for {log_name_base}')
df

### PyOx Atom Labeling

In [ ]:
# finds which N is in the pyridine and adjusts atom numbering as needed 
sdf_dir = '/Users/theresewild/Sigman Group Dropbox/Therese Wild/NN_Library/dft_library_all/fluoride_library_ligands/fluoride_logs/biim_allS'
seen_prefixes = set()

for i,row in df.iterrows():
    
    log_name_base = row['log_name']
    N1_index = row['N1'] - 1 #subtract 1 because mol is 0 indexed
    N2_index = row['N2'] - 1
    C1_index = row['C1'] - 1
    C2_index = row['C2'] - 1
    
    sdf_name = log_name_base + '.sdf'

    sdf_path = os.path.join(sdf_dir,sdf_name)
    try:
        suppl = Chem.SDMolSupplier(sdf_path, removeHs=False)
        mol = next((m for m in suppl if m is not None), None)
    except:
        print (f'bad input file for {sdf_path}')
    ringsize_N1_is_6 = mol.GetAtomWithIdx(N1_index).IsInRingSize(6)
    ringsize_N1_is_5 = mol.GetAtomWithIdx(N1_index).IsInRingSize(5)

    ringsize_N2_is_6 = mol.GetAtomWithIdx(N2_index).IsInRingSize(6)
    ringsize_N2_is_5 = mol.GetAtomWithIdx(N2_index).IsInRingSize(5)

    C2_atomic_num = mol.GetAtomWithIdx(C2_index).GetAtomicNum()
    C1_atomic_num = mol.GetAtomWithIdx(C1_index).GetAtomicNum()
    N2_atomic_num = mol.GetAtomWithIdx(N2_index).GetAtomicNum()
    N1_atomic_num = mol.GetAtomWithIdx(N1_index).GetAtomicNum()

    if ringsize_N1_is_6 is True and ringsize_N2_is_6 is True: #if it is a pyNx ligand just do by hand 
        print (f'both 6 membered rings, correct {log_name_base} manually')
    if ringsize_N1_is_6 is False and ringsize_N2_is_6 is False and ringsize_N2_is_5 is True: # if there are 2 5 membered and no 6 membered ring
        print (f'incorrect substructure for {log_name_base}')
    elif C2_atomic_num != 6 or C1_atomic_num != 6 or N2_atomic_num !=7 or N1_atomic_num !=7:
        print (f'error in atom numbering for {log_name_base}')  

    elif ringsize_N1_is_6 is True and ringsize_N2_is_5 is True and ringsize_N2_is_6 is False: # is N1 is already pyridine, df is correct as is 
        # write back corrected values (+1 because dataframe is 1-based)
        df.loc[i, 'N1'], df.loc[i, 'N2']  = N1_index + 1, N2_index + 1
        df.loc[i, 'C1'], df.loc[i, 'C2'] = C1_index + 1, C2_index + 1

    elif ringsize_N2_is_6 is True and ringsize_N1_is_5 is True and ringsize_N1_is_6 is False:
        df.loc[i, 'N1'], df.loc[i, 'N2'] = N2_index + 1, N1_index + 1
        df.loc[i, 'C1'], df.loc[i, 'C2'] = C2_index + 1, C1_index + 1
        
df

In [ ]:
df.to_excel('pyox_atom_labeling.xlsx', index=False)

### All Other Atom Labeling

#### Property Collection Functions

In [ ]:
def get_geom(streams):
    geom = []
    for item in streams[-1][16:]:
        if item == "":
            break
        geom.append([item.split(",")[0],float(item.split(",")[-3]),float(item.split(",")[-2]),float(item.split(",")[-1])])
    return(geom)

def get_outstreams(log): 
    streams = []
    starts,ends = [],[]
    error = ""
    an_error = True
    try:
        with open(log+".log") as f:
            loglines = f.readlines()
    except:
        with open(log+".LOG") as f:
            loglines = f.readlines()
            
    for line in loglines[::-1]:
        if "Normal termination" in line:
            an_error = False
        if an_error:
            error = "****Failed or incomplete jobs for " + log + ".log"        
            
    for i in range(len(loglines)):
        if "1\\1\\" in loglines[i]:
            starts.append(i)
        if "@" in loglines[i]:
            ends.append(i)
       if "Normal termination" in loglines[i]:
           error = ""
            
            
    if len(starts) != len(ends) or len(starts) == 0: #probably redundant
        error = "****Failed or incomplete jobs for " + log + ".log"
        return(streams,error)
    for i in range(len(starts)):
        tmp = ""
        for j in range(starts[i],ends[i]+1,1):
            tmp = tmp + loglines[j][1:-1]
        streams.append(tmp.split("\\"))
    return(streams,error)

def get_filecont(log): #gets the entire job output
    error = "" #default unless "normal termination" is in file
    an_error = True
    with open(log+".log") as f:
        loglines = f.readlines()
    for line in loglines[::-1]:
        if "Normal termination" in line:
            an_error = False
        if an_error:
            error = "****Failed or incomplete jobs for " + log + ".log"
    return(loglines, error)

def get_vbur_quadrants_only(dataframe, a1, ex1, ex2, z1, z2, radius):
    """
    Uses Morfeus to calculate %Vbur at a single radius for atom (a1) in df.
    """
    atom = str(a1)
    atom_ex1 = str(ex1)
    atom_ex2 = str(ex2)
    atom_z1 = str(z1)
    atom_z2 = str(z2)

    rows = []  # Collect result rows here

    for index, row in dataframe.iterrows():
        log_file = row['log_name']
        atom1 = row[atom]      
        exclude1 = row[atom_ex1]
        exclude2 = row[atom_ex2]
        zaxis1 = row[atom_z1]
        zaxis2 = row[atom_z2]
        xzatom = row[atom_z2]

        streams, error = get_outstreams(log_file)
        if error:
            print(error)
            row_i = {
                f'%Vbur_{atom}_quadrant_+,+': "no data",
                f'%Vbur_{atom}_quadrant_–,+': "no data",
                f'%Vbur_{atom}_quadrant_–,–': "no data",
                f'%Vbur_{atom}_quadrant_+,–': "no data",
            }
            rows.append(row_i)
            continue

        log_coordinates = get_geom(streams)
        elements = np.array([entry[0] for entry in log_coordinates])
        coordinates = np.array([entry[1:] for entry in log_coordinates], dtype=float)

        vbur = BuriedVolume(
            elements,
            coordinates,
            int(atom1),
            radius=radius,
            include_hs=True,
            excluded_atoms=[exclude1, exclude2],
            z_axis_atoms=[int(zaxis1), int(zaxis2)],
            xz_plane_atoms=[int(xzatom)],
        )

        vbur.octant_analysis()
        bv_quadrants = vbur.quadrants["percent_buried_volume"]

        row_i = {
            f'%Vbur_{atom}_quadrant_+,+': bv_quadrants[1],
            f'%Vbur_{atom}_quadrant_–,+': bv_quadrants[2],
            f'%Vbur_{atom}_quadrant_–,–': bv_quadrants[3],
            f'%Vbur_{atom}_quadrant_+,–': bv_quadrants[4],
        }

        rows.append(row_i)
    vbur_quadoct_dataframe = pd.DataFrame(rows)
    print("Buried volume quadrants and octants function has completed")
    return pd.concat([dataframe.reset_index(drop=True), vbur_quadoct_dataframe], axis=1)

def get_goodvibes_e(dataframe, temp):
    rows = []  
    options = gv.GVOptions()
    options.spc = 'link'
    options.temperature = temp
    log = io.Logger("Goodvibes", 'output', False)

    for index, row in dataframe.iterrows():
        try:
            log_file = row['log_name']
            file_data = io.getoutData(str(log_file) + ".log", options)

            options.freq_scale_factor = False
            level_of_theory = [file_data.functional + '/' + file_data.basis_set]
            options.freq_scale_factor, options.mm_freq_scale_factor = gv.get_vib_scale_factor(
                level_of_theory, options, log
            )

            bbe_val = thermo.calc_bbe(file_data, options)
            properties = [
                'sp_energy', 'zpe', 'enthalpy', 'entropy', 
                'qh_entropy', 'gibbs_free_energy', 'qh_gibbs_free_energy'
            ]
            vals = [getattr(bbe_val, k) for k in properties]

            row_i = {
                'E_spc (Hartree)': vals[0],
                'ZPE(Hartree)': vals[1],
                'H_spc(Hartree)': vals[2],
                'T*S': vals[3] * options.temperature,
                'T*qh_S': vals[4] * options.temperature,
                'G(T)_spc(Hartree)': vals[5],
                'qh_G(T)_spc(Hartree)': vals[6],
                'T': options.temperature
            }

        except Exception as e:
            print(f"\n**** Unable to acquire GoodVibes energies for: {row.get('log_name', 'unknown')}.log")
            row_i = {
                'E_spc (Hartree)': "no data",
                'ZPE(Hartree)': "no data",
                'H_spc(Hartree)': "no data",
                'T*S': "no data",
                'T*qh_S': "no data",
                'G(T)_spc(Hartree)': "no data",
                'qh_G(T)_spc(Hartree)': "no data",
                'T': "no data"
            }

        rows.append(row_i)
    e_dataframe = pd.DataFrame(rows)
    return pd.concat([dataframe.reset_index(drop=True), e_dataframe], axis=1)

#### Collect Property For Atom Re-Labeling

In [ ]:
from morfeus import BuriedVolume
metal_atom = "Ni"
#---------------GoodVibes Energies---------------
#uses the GoodVibes 2021 Branch (Jupyter Notebook Compatible)
#calculates the quasi harmonic corrected G(T) and single point corrected G(T) as well as other thermodynamic properties
#inputs: dataframe, temperature
df = get_goodvibes_e(df, 298.15)

# -------------Vbur quadrants ------
#uses the MORFEUS buried volume function and splits into quadrants and octants
#define: df, center of sphere, excluded atom1, excluded atom2, z-axis atom1, z-axis atom2
df = get_vbur_quadrants_only(df, a1=metal_atom, ex1="-H1", ex2="-H2", z1="N1", z2="N2", radius=6.5)

df['north_hemisphere'] = df['%Vbur_Ni_quadrant_+,+'] + df['%Vbur_Ni_quadrant_+,–']
df['south_hemisphere'] = df['%Vbur_Ni_quadrant_–,–'] + df['%Vbur_Ni_quadrant_–,+']

df.to_excel('properties_for_atom_map.xlsx', index=False)

#### Relabeling

In [ ]:
df = pd.read_excel('properties_for_atom_map.xlsx')
prefix = "Lig" 
suffix = "_"

energy_col_header = "G(T)_spc(Hartree)"

compound_list = []
    
for index, row in df.iterrows():
    log_file = row['log_name'] 
    prefix_and_compound = log_file.split(str(suffix))
    compound = prefix_and_compound[0].split(str(prefix)) 
    compound_list.append(compound[1])

compound_list = list(set(compound_list))
compound_list.sort()

In [ ]:
property_to_compare = ["north_hemisphere", "south_hemisphere"] # if north hemisphere is bigger than N2 is correctly assigned and is south hemisphere is bigger it needs to swap (i.e. north hemisphere is N2 and south hemisphere is N1)
dict_of_N2 = {}
dict_of_N1 = {}
dict_of_C2 = {}
dict_of_C1 = {}

for compound in compound_list: 
    base = f"{prefix}{compound}"
    pattern = rf"^{base}(_|$)"        
    compounddf = df[df["log_name"].str.match(pattern)]
    compounddf = compounddf.reset_index(drop = True)  
    compounddf["∆G(Hartree)"] = compounddf[energy_col_header] - compounddf[energy_col_header].min()
    low_e_index = compounddf[compounddf["∆G(Hartree)"] == 0].index.tolist()
    prop_1 = compounddf[str(property_to_compare[0])][low_e_index[0]] #first property listed above, should be N2 (north )
    prop_2 = compounddf[str(property_to_compare[1])][low_e_index[0]] #second property listed above, should be N1
    if prop_1 >= prop_2: # if the N2 property is greater than the N1 property, N2 should stay as N2 and N1 should stay as N1 - and then C2 and C1 stay as is 
        N2 = compounddf["N2"][low_e_index[0]] #N2 is bigger
        C2 = compounddf["C2"][low_e_index[0]] #C2 has to move with N2
        N1 = compounddf["N1"][low_e_index[0]] #N1 is smaller
        C1 = compounddf["C1"][low_e_index[0]] #C1 has to move with N1
    elif prop_1 < prop_2: # if N1 is larger than N2, reassign N1 as N2 and C1 as C2 
        N2 = compounddf["N1"][low_e_index[0]]
        C2 = compounddf["C1"][low_e_index[0]]
        N1 = compounddf["N2"][low_e_index[0]]
        C1 = compounddf["C2"][low_e_index[0]]

    #for every conformer the compound has, give the results from the low E as the value and the name of the log file as the key for a dictionary we will later add to the original dataframe
    for index, row in compounddf.iterrows():
        key = row['log_name']
        dict_of_N2[key] = N2
        dict_of_N1[key] = N1
        dict_of_C2[key] = C2
        dict_of_C1[key] = C1
        

df['N2'] = df['log_name'].map(dict_of_N2)
df['N1'] = df['log_name'].map(dict_of_N1)
df['C2'] = df['log_name'].map(dict_of_C2)
df['C1'] = df['log_name'].map(dict_of_C1)

In [ ]:
df.to_excel('modeling_atom_map_with_N2_as_bigger.xlsx')

# Import a manually-generated atom mapping dataframe

In [ ]:
atom_map_df = pd.read_excel('modeling_atom_map_with_N2_as_bigger.xlsx','Sheet1',index_col=0,header=0,engine='openpyxl')
atom_map_df.reset_index(inplace=True, drop=True)
atom_map_df.drop(columns=[
       '%Vbur_Ni_quadrant_+,+', '%Vbur_Ni_quadrant_+,–',
       '%Vbur_Ni_quadrant_–,+', '%Vbur_Ni_quadrant_–,–', 'north_hemisphere',
       'south_hemisphere'], inplace=True)

df = atom_map_df

# Define Properties to Collect

## Copy and modify available property functions above to customize

In [ ]:
#---------------GoodVibes Engergies---------------
#uses the GoodVibes 2021 Branch (Jupyter Notebook Compatible)
#calculates the quasi harmonic corrected G(T) and single point corrected G(T) as well as other thermodynamic properties
# #inputs: dataframe, temperature
# df = gp.get_goodvibes_e(df, 298.15)

#---------------Frontier Orbitals-----------------
#E(HOMO), E(LUMO), mu(chemical potential or negative of molecular electronegativity), eta(hardness/softness), omega(electrophilicity index)
df = gp.get_frontierorbs(df)

#---------------Polarizability--------------------
#Exact polarizability
df = gp.get_polarizability(df)

#---------------Dipole----------------------------
#Total dipole moment magnitude in Debye
df = gp.get_dipole(df)

#---------------SASA------------------------------
#Uses morfeus to calculat sovlent accessible surface area and the volume under the SASA
df = gp.get_SASA(df)

#---------------NBO-------------------------------
#natural charge from NBO
#requires the Gaussian keyword = "pop=nbo7" in the .com file
nbo_list = ["C1", "C2", "N1", "N2"]
df = gp.get_nbo(df, nbo_list) 

#---------------NMR-------------------------------
# isotropic NMR shift
# requires the Gaussian keyword = "nmr=giao" in the .com file
nmr_list = ["C1", "C2", "N1", "N2"]
df = gp.get_nmr(df, nmr_list) 

#---------------Distance--------------------------
#distance between 2 atoms
dist_list_of_lists = [["N1", "Ni"], ["N2", "Ni"], ["N2", "C2"], ["N1", "C1"]]
df = gp.get_distance(df, dist_list_of_lists) 

#---------------Angle-----------------------------
#angle between 3 atoms
angle_list_of_lists = [["N1", "C1", "C2"], ["N2", "C2", "C1"]]
df = gp.get_angles(df, angle_list_of_lists) 

# #--------------Vbur Scan bisphosphines-----------------
# #this is the same as the above MORFEUS Vbur Scan function, except it removes both Cls on the metal center
# #need to update this function to make it more general...
a_list = ['Ni']
df = gp.get_vbur_scan(df, a_list, 2, 4, .5)

#---------------Vbur Scan-------------------------
#uses morfeus to calculate the buried volume at a series of radii (including hydrogens)
#inputs: dataframe, list of atoms, start_radius, end_radius, and step_size
#if you only want a single radius, put the same value for start_radius and end_radius (keep step_size > 0)
vbur_list = ["N1", "N2", "C2", "C1"]
df = gp.get_vbur_scan_no_metal(df, vbur_list, 3, 5, 0.5)
    
#---------------Sterimol morfeus------------------
#uses morfeus to calculate Sterimol L, B1, and B5 values
#NOTE: this is much faster than the corresponding DBSTEP function (recommendation: use as default/if you don't need Sterimol2Vec)
sterimol_list_of_lists = [["N1", "Ni"], ["C1", "N1"], ["N2", "C2"], ["N2", "Ni"], ["Ni", "N1"], ["N1", "C1"], ["C2", "N2"], ["Ni", "N2"]]
df = gp.get_sterimol_morfeus(df, sterimol_list_of_lists) 

#---------------Buried Sterimol-------------------
#uses morfeus to calculate Sterimol L, B1, and B5 values within a given sphere of radius r_buried
#atoms outside the sphere + 0.5 vdW radius are deleted and the Sterimol vectors are calculated
#for more information: https://kjelljorner.github.io/morfeus/sterimol.html
#inputs: dataframe, list of atom pairs, r_buried
sterimol_list_of_lists = [["N1", "Ni"], ["C1", "N1"], ["N2", "C2"], ["N2", "Ni"], ["Ni", "N1"], ["N1", "C1"], ["C2", "N2"], ["Ni", "N2"]]
df = gp.get_buried_sterimol(df, sterimol_list_of_lists, 5.5) 

#---------------Hirshfeld-------------------------
#Hirshfeld charge, CM5 charge, Hirshfeld atom dipole
#requires the Gaussian keyword = "pop=hirshfeld" in the .com file
a_list = ['N1', 'N2', 'C1', 'C2']
df = gp.get_hirshfeld(df, a_list) 

#-------------Vbur quadrants ------
#uses the MORFEUS buried volume function and splits into quadrants and octants
#define: df, center of sphere, excluded atom1, excluded atom2, z-axis atom1, z-axis atom2
df = gp.get_vbur_quadrants_octants(df, a1 = 'Ni', ex1 = "-H1", ex2 = "-H2", z1 = "N1", z2 = "N2", radius=5.0)

# -------------Bidentate ligand bite angle------------------
# this give the same information as the angle function, but the output is cleaner
# define: df, metal atom, donor atom 1, donor atom 2
df = gp.get_bite_angle(df, 'Ni', "N1", "N2")

df = gp.get_visible_volume(df, 'Ni', ['-H1', '-H2'])

pd.options.display.max_columns = None
df.to_excel('properties_raw.xlsx')

# Post-processing

## User input for data processing

In [ ]:
prefix = "Lig" 
suffix = "_"
atom_columns_to_drop = ["C2", "C1", "N1", "Ni", "N2", "-H1", "-H2"]
energy_col_header = "G(T)_spc(Hartree)"

In [ ]:
df = pd.read_excel('properties_raw.xlsx','Sheet1',header=0,engine='openpyxl')

## Generating a list of compounds that have conformational ensembles

In [ ]:
compound_list = []
    
for index, row in df.iterrows():
    log_file = row['log_name'] 
    prefix_and_compound = log_file.split(str(suffix)) 
    compound = prefix_and_compound[0].split(str(prefix)) 
    compound_list.append(compound[1])

compound_list = list(set(compound_list))
compound_list.sort() 
print(compound_list)

['1953', '1954', '1955', '1956', '1957', '1958', '2037', '2039', '2055', '2056', '2098', '2102', '2145', '2207', '2226', '2285', '2298', '2311', '2312', '2314', '2315', '2316', '2318', '459', '57', '58', '60', '61', '62', '63', '64', '649', '65', '66', '67', '68', '69', '697', '71', '719', '72', '723', '73', '737', '747', '75', '76', '768', '769', '77', '795', '80', '81', '82', '821', '83', '84', '844', '85', '881', '894', '908', '920', '982', '983']


## Post-processing to get properties for each compound

In [ ]:
all_df_master = pd.DataFrame(columns=[])
properties_df_master = pd.DataFrame(columns=[])

for compound in compound_list: 
    base = f"{prefix}{compound}"
    pattern = rf"^{base}(_|$)"       
    valuesdf = df[df["log_name"].str.match(pattern)]
    valuesdf = valuesdf.drop(columns = atom_columns_to_drop)
    valuesdf = valuesdf.reset_index(drop = True)  
   
    #define columns that won't be included in summary properties or are treated differently because they don't make sense to Boltzmann average
    non_boltz_columns = ["G(Hartree)","∆G(Hartree)","∆G(kcal/mol)", "e^(-∆G/RT)","Mole Fraction"] #don't boltzman average columns containing these strings in the column label
    reg_avg_columns = ['CPU_time_total(hours)', 'Wall_time_total(hours)'] #don't boltzmann average these either, we average them in case that is helpful
    gv_extra_columns = ['E_spc (Hartree)', 'H_spc(Hartree)', 'T', 'T*S', 'T*qh_S', 'ZPE(Hartree)', 'qh_G(T)_spc(Hartree)', "G(T)_spc(Hartree)"]
    gv_extra_columns.remove(str(energy_col_header))
    
    #calculate the summary properties based on all conformers (Boltzmann Average, Minimum, Maximum, Boltzmann Weighted Std)
    valuesdf["∆G(Hartree)"] = valuesdf[energy_col_header] - valuesdf[energy_col_header].min()
    #print (valuesdf["∆G(Hartree)"])
    valuesdf["∆G(kcal/mol)"] = valuesdf["∆G(Hartree)"] * 627.5
    valuesdf["e^(-∆G/RT)"] = np.exp((valuesdf["∆G(kcal/mol)"] * -1000) / (1.987204 * 298.15)) #R is in cal/(K*mol)
    valuesdf["Mole Fraction"] = valuesdf["e^(-∆G/RT)"] / valuesdf["e^(-∆G/RT)"].sum()
    values_boltz_row = []
    values_min_row = []
    values_max_row = []
    values_boltz_stdev_row =[]
    values_range_row = []
    values_exclude_columns = []
    
    for column in valuesdf:
        if "log_name" in column:
            values_boltz_row.append("Boltzmann Averages")
            values_min_row.append("Ensemble Minimum")
            values_max_row.append("Ensemble Maximum")
            values_boltz_stdev_row.append("Boltzmann Standard Deviation")
            values_range_row.append("Ensemble Range")
            values_exclude_columns.append(column) #used later to build final dataframe
        elif any(phrase in column for phrase in non_boltz_columns) or any(phrase in column for phrase in gv_extra_columns):
            values_boltz_row.append("")
            values_min_row.append("")
            values_max_row.append("")
            values_boltz_stdev_row.append("")
            values_range_row.append("")
        elif any(phrase in column for phrase in reg_avg_columns):
            values_boltz_row.append(valuesdf[column].mean()) #intended to print the average CPU/wall time in the boltz column
            values_min_row.append("")
            values_max_row.append("")
            values_boltz_stdev_row.append("")
            values_range_row.append("")
        else:
            valuesdf[column] = pd.to_numeric(valuesdf[column]) #to hopefully solve the error that sometimes occurs where the float(Mole Fraction) cannot be mulitplied by the string(property)
            values_boltz_row.append((valuesdf[column] * valuesdf["Mole Fraction"]).sum())
            values_min_row.append(valuesdf[column].min())
            values_max_row.append(valuesdf[column].max())
            values_range_row.append(valuesdf[column].max() - valuesdf[column].min())

        
            # this section generates the weighted std deviation (weighted by mole fraction) 
            # formula: https://www.statology.org/weighted-standard-deviation-excel/
    
            boltz = (valuesdf[column] * valuesdf["Mole Fraction"]).sum() #number
            #print (boltz)
            delta_values_sq = []
    
            #makes a list of the "deviation" for each conformer           
            for index, row in valuesdf.iterrows(): 
                value = row[column]
                delta_value_sq = (value - boltz)**2
                delta_values_sq.append(delta_value_sq)
            
            #w is list of weights (i.e. mole fractions)
            w = list(valuesdf["Mole Fraction"])
            wstdev = np.sqrt( (np.average(delta_values_sq, weights=w)) / (((len(w)-1)/len(w))*np.sum(w)) )
            if len(w) == 1: #if there is only one conformer in the ensemble, set the weighted standard deviation to 0 
                wstdev = 0
            #np.average(delta_values_sq, weights=w) generates sum of each (delta_value_sq * mole fraction)
            
            values_boltz_stdev_row.append(wstdev)
            #print (wstdev)
            
    valuesdf.loc[len(valuesdf)] = values_boltz_row
    valuesdf.loc[len(valuesdf)] = values_boltz_stdev_row
    valuesdf.loc[len(valuesdf)] = values_min_row
    valuesdf.loc[len(valuesdf)] = values_max_row
    valuesdf.loc[len(valuesdf)] = values_range_row

    #final output format is built here:
    explicit_order_front_columns = ["log_name", energy_col_header,"∆G(Hartree)","∆G(kcal/mol)","e^(-∆G/RT)","Mole Fraction"]
    
    #reorders the dataframe using front columns defined above
    valuesdf = valuesdf[explicit_order_front_columns + [col for col in valuesdf.columns if col not in explicit_order_front_columns and col not in values_exclude_columns]]
    
    #determine the index of the lowest energy conformer
    low_e_index = valuesdf[valuesdf["∆G(Hartree)"] == 0].index.tolist()
    
    #copy the row to a new_row with the name of the log changed to Lowest E Conformer
    new_row = valuesdf.loc[low_e_index[0]]
    new_row['log_name'] = "Lowest E Conformer"   
    valuesdf =  valuesdf.append(new_row, ignore_index=True)
    
    #appends the frame to the master output
    all_df_master = pd.concat([all_df_master, valuesdf])
    
    #drop all the individual conformers
    dropindex = valuesdf[valuesdf["log_name"].str.match(pattern)].index
    valuesdf = valuesdf.drop(dropindex)
    valuesdf = valuesdf.reset_index(drop = True)
    
    #drop the columns created to determine the mole fraction and some that 
    valuesdf = valuesdf.drop(columns = explicit_order_front_columns)
    try:
        valuesdf = valuesdf.drop(columns = gv_extra_columns)
    except:
        pass
    try:
        valuesdf = valuesdf.drop(columns = reg_avg_columns)
    except:
        pass
        
#---------------------THIS MAY NEED TO CHANGE DEPENDING ON HOW YOU LABEL YOUR COMPOUNDS------------------------------  
    compound_name = prefix + str(compound) 
#--------------------------------------------------------------------------------------------------------------------      

    properties_df = pd.DataFrame({'Compound_Name': [compound_name]})
   
    for (columnName, columnData) in valuesdf.iteritems():
        properties_df[str(columnName) + "_Boltz"] = [columnData.values[0]]
        properties_df[str(columnName) + "_Boltz_stdev"] = [columnData.values[1]]
        properties_df[str(columnName) + "_min"] = [columnData.values[2]]
        properties_df[str(columnName) + "_max"] = [columnData.values[3]]
        properties_df[str(columnName) + "_range"] = [columnData.values[4]]
        properties_df[str(columnName) + "_low_E"] = [columnData.values[5]]
        
    properties_df_master = pd.concat([properties_df_master, properties_df], axis = 0)

all_df_master = all_df_master.reset_index(drop = True)
properties_df_master = properties_df_master.reset_index(drop = True)


ValueError: Unable to parse string "no data" at position 8

### Save to Microsoft Excelᵀᴹ 

In [ ]:
all_df_master.to_excel('All_Conformer_Properties_hydride_ligands.xlsx', index = False)
properties_df_master.to_excel('Summary_Properties_hydride_ligands.xlsx', index = False)

In [ ]:
df = pd.read_excel('Summary_Properties_hydride_ligands.xlsx','Sheet1',index_col=0,header=0,engine='openpyxl')

suffixes_to_drop = ['_stdev', '_range']
pattern = '|'.join(suffixes_to_drop)

columns_to_drop = df.filter(regex=pattern).columns
df = df.drop(columns_to_drop, axis=1)

df.to_excel ('all_props_hydride_ligands.xlsx')